In [1]:
import pandas as pd
import os
import sys
dir1 = os.path.abspath(os.path.join(os.getcwd(), '../../analysisFunctions'))
sys.path.insert(0, dir1)
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.api as sm
from statsmodels.formula.api import ols
from statsmodels.stats.multicomp import pairwise_tukeyhsd
import re
from machine_learning import *
from hyperion_utils import *

/home/idies/miniconda3/lib/python3.9/site-packages/xgboost/core.py:265: FutureWarning: Your system has an old version of glibc (< 2.28). We will stop supporting Linux distros with glibc older than 2.28 after **May 31, 2025**. Please upgrade to a recent Linux distro (with glibc 2.28+) to use future versions of XGBoost.
Note: You have installed the 'manylinux2014' variant of XGBoost. Certain features such as GPU algorithms or federated learning are not available. To use these features, please upgrade to a recent Linux distro with glibc 2.28+, and install the 'manylinux_2_28' variant.
  warnings.warn(


In [2]:
myDfs = readDfs()

In [3]:
print(myDfs['Description_tables_variables'][['Tables', 'Variable', 'LabelTranslated', 'Label']].to_string())

         Tables                          Variable                                                                                             LabelTranslated                                                                                                 Label
0           ADL                            SUBJID                                                                            Subject Identifier for the Study                                                                      Subject Identifier for the Study
1           ADL                             ADL_0                                                                             Has the ADL score was completed                                                                      Le score ADL a t-il été complété
2           ADL                        ADL_1_TEMP                                                               Body hygiene (intermediate field for seizure)                                                  Hygiène corpo

In [4]:
bio_data_columns = list(myDfs['Description_tables_variables']['Variable'].iloc[52:76])
bio_data_columns.append('SUBJID')
bio_data_columns_descr = list(myDfs['Description_tables_variables']['Label'].iloc[52:76])
bio_visit_column = 'VISIT'
cpc_column = 'CPC_SC3'
group_colums = ['V0_BRAS2', 'groupe']
ds_columns = 'DS_DATA_REFUS'
ecg_columns = list(myDfs['Description_tables_variables']['Variable'].iloc[137:154])
ecg_columns_descr = list(myDfs['Description_tables_variables']['LabelTranslated'].iloc[137:154])
j0_drop_columns = list(myDfs['Description_tables_variables']['Variable'].iloc[362:384])
j0_reflex_columns = list(myDfs['Description_tables_variables']['Variable'].iloc[273:285])
j0_reflex_columns_descr = list(myDfs['Description_tables_variables']['LabelTranslated'].iloc[273:285])
sofa_columns = list(myDfs['Description_tables_variables']['Variable'].iloc[586:593])
sofa_columns.append('SUBJID')
ei_columns = list(myDfs['Description_tables_variables']['Variable'].iloc[158:168])
ei_columns = ei_columns + (['SUBJID', 'EI_ARYTHMI', 'EI_ANTIEPILEPTIQ'])

In [6]:
myPredictorsDf = myDfs['ds'][myDfs['ds']['DS_DATA_REFUS'] != 1][['SUBJID', 'DS_DATA_REFUS']]
myPredictorsDf = myPredictorsDf.merge(myDfs['bras_rando'], on='SUBJID')
myPredictorsDf = myPredictorsDf[~myPredictorsDf['groupe'].isna()]
myPredictorsDf = myPredictorsDf.merge(myDfs['cpc'][['SUBJID', cpc_column]], on='SUBJID')
myPredictorsDf = myPredictorsDf.merge(myDfs['j0'], on='SUBJID')
myPredictorsDf = myPredictorsDf.merge(myDfs['bio'][myDfs['bio']['VISIT'] == 'J0'][bio_data_columns], on='SUBJID')
myPredictorsDf = myPredictorsDf.merge(myDfs['ecg'][myDfs['ecg']['VISIT'] == 'J0'][ecg_columns], on='SUBJID')
myPredictorsDf = myPredictorsDf.merge(myDfs['sofa'][myDfs['sofa']['VISIT'] == 'J0'][sofa_columns], on='SUBJID')
myPredictorsDf = myPredictorsDf.merge(myDfs['ei'][myDfs['ei']['CRFSTATUS'] == 'J0'][ei_columns], on='SUBJID')
# myPredictorsDf = myPredictorsDf.merge(myDfs['adl'][['SUBJID', 'ADL_SC']], on='SUBJID')
myPredictorsDf = myPredictorsDf.merge(myDfs['barthel'][['SUBJID', 'BARTHEL_SC']], on='SUBJID')
# myPredictorsDf = myPredictorsDf.merge(myDfs['mms'][['SUBJID', 'MMS_SC']], on='SUBJID')
myDay0SofaSc = myDfs['sofa'][(myDfs['sofa']['VISIT'] == 'J0')][['SUBJID', 'SOFA_SC']]
myDay7SofaSc = myDfs['sofa'][(myDfs['sofa']['VISIT'] == 'J7')][['SUBJID', 'SOFA_SC']]
myDay7SofaSc.rename(columns={'SOFA_SC': 'SOFA_SC7'}, inplace=True)
myDay0SofaSc.rename(columns={'SOFA_SC': 'SOFA_SC1'}, inplace=True)
myPredictorsDf = myPredictorsDf.merge(myDay7SofaSc, on='SUBJID')
myPredictorsDf = myPredictorsDf.merge(myDay0SofaSc, on='SUBJID')
myPredictorsDf = myPredictorsDf.merge(myDfs['ds'][['DS_DC', 'DS_DT_DC', 'SUBJID']], on='SUBJID')
myPredictorsDf = myPredictorsDf.merge(myDfs['date'][['DT_J0', 'SUBJID']], on='SUBJID')
myPredictorsDf['DAYS_ALIVE_30'] = (pd.to_datetime(myPredictorsDf['DS_DT_DC'], format='%d/%m/%Y') - pd.to_datetime(pd.to_datetime(myPredictorsDf['DT_J0']).dt.date)).dt.days
myPredictorsDf['DAYS_ALIVE_30'].fillna(30, inplace=True)
myPredictorsDf.loc[myPredictorsDf['DAYS_ALIVE_30'] > 30, 'DAYS_ALIVE_30'] = 30

In [7]:
myPredictorsDf.drop(columns=['DS_DT_DC', 'DT_J0', 'DS_DATA_REFUS', 'Unnamed: 0_x', 'Unnamed: 0_y', 'V0_BRAS2', 'DATEDERANDOAPRENDREENCOMPTEPOURA', 'J0_PANCARTE', 'J0_PANCARTET1', 'J0_NAIS_M', 'J0_NAIS_A', 'J0_DNAIS',  'J0_CAUSE2_ACR_P', 'J0_H_37T1', 'VISIT'], inplace=True)
myPredictorsDf.drop(columns=j0_drop_columns, inplace=True)
myPredictorsDf['CPC_SC3'].fillna(5, inplace=True)

In [8]:
myPredictorsDf['CPC12'] = ((myPredictorsDf['CPC_SC3'] == 1) | (myPredictorsDf['CPC_SC3'] == 2)).astype(int)
myPredictorsDf['SEX'] = (myPredictorsDf['J0_SEX'] == 'Homme').astype(int)


In [11]:
myPredictorsDf.to_csv('predictorsDf.csv', index=False)

/home/idies/miniconda3/lib/python3.9/site-packages/pandas/core/internals/blocks.py:2323: RuntimeWarning: invalid value encountered in cast
  values = values.astype(str)


In [10]:
myPredictorsDf

,SUBJID,groupe,CPC_SC3,J0_SEX,J0_TAILLE,J0_POIDS,J0_BMI,J0_AGE,J0_PAS,J0_PAD,...,EI_CONVULS,EI_ARYTHMI,EI_ANTIEPILEPTIQ,BARTHEL_SC,SOFA_SC7,SOFA_SC1,DS_DC,DAYS_ALIVE_30,CPC12,SEX
0,1001,1.0,5.0,Femme,150.0,70.0,31.1,84.0,95.0,42.0,...,NaN,NaN,NaN,NaN,7.0,11.0,1.0,7.0,0,0
1,1002,0.0,5.0,Femme,163.0,94.0,35.4,75.0,130.0,68.0,...,NaN,NaN,NaN,NaN,NaN,14.0,1.0,-139.0,0,0
2,1003,0.0,5.0,Homme,167.0,78.0,28.0,75.0,144.0,80.0,...,10.0,0.0,0.0,NaN,NaN,11.0,1.0,3.0,0,1
3,1004,0.0,3.0,Femme,159.0,34.0,13.4,50.0,143.0,82.0,...,NaN,NaN,NaN,80.0,7.0,16.0,0.0,30.0,0,0
4,1005,0.0,5.0,Homme,179.0,105.0,32.8,63.0,195.0,104.0,...,NaN,NaN,NaN,NaN,NaN,9.0,1.0,2.0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
576,26010,0.0,5.0,Homme,167.0,63.0,22.6,70.0,114.0,52.0,...,0.0,0.0,NaN,NaN,NaN,18.0,1.0,3.0,0,1
577,26011,1.0,5.0,Homme,NaN,82.0,NaN,68.0,127.0,71.0,...,NaN,NaN,NaN,NaN,2.0,12.0,0.0,30.0,0,1
578,26012,0.0,5.0,Homme,167.0,72.0,25.8,78.0,102.0,57.0,...,NaN,NaN,NaN,NaN,6.0,11.0,1.0,30.0,0,1
579,26014,0.0,5.0,Femme,146.0,71.0,33.3,82.0,226.0,104.0,...,1.0,0.0,1.0,NaN,NaN,10.0,1.0,2.0,0,0
